## 🧬 NGS Mini‑Pipeline Training

## 📄 What is a VCF file?

In the previous notebook we ran `bcftools call` and produced a file called `variants.vcf`.

The **VCF (Variant Call Format)** is the standard file format for storing genetic variants. It is used by every major variant caller and is the format in which clinical bioinformatics results are reported, filtered, and annotated.

Understanding VCF is essential for anyone working in clinical genomics.

---

### 🗂️ Overall structure

A VCF file has three sections:

| Section | Starts with | Purpose |
|---|---|---|
| Meta-information | `##` | Describes the file: tools used, reference, definitions of INFO/FORMAT fields |
| Column header | `#CHROM` | Names the 10 columns of the data section |
| Data lines | A base (A/C/G/T) | One line per variant |

Let's look at the full file we generated:

In [ ]:
!cat ../fastq_files/variants.vcf

---

## 📋 Section 1 — The `##` meta-information lines

Every `##` line defines something about the file. The most important ones are:

**`##fileformat`** — the VCF version:
```
##fileformat=VCFv4.2
```

**`##FILTER`** — defines filter labels used in the FILTER column:
```
##FILTER=<ID=PASS,Description="All filters passed">
```

**`##INFO`** — defines the key-value pairs that appear in the INFO column. For example:
```
##INFO=<ID=DP,Number=1,Type=Integer,Description="Raw read depth">
##INFO=<ID=AF,Number=A,Type=Float,Description="Allele frequency">
```

**`##FORMAT`** — defines the fields in the per-sample column. For example:
```
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Read depth">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths">
```

**`##contig`** — describes the reference sequences:
```
##contig=<ID=chr1:100000-101000,length=1000>
```

These definitions are critical — without them, you cannot interpret the INFO and FORMAT fields correctly.

---

## 📋 Section 2 — The column header line

The line starting with `#CHROM` names the 10 mandatory columns:

```
#CHROM  POS  ID  REF  ALT  QUAL  FILTER  INFO  FORMAT  SAMPLE
```

If there are multiple samples, additional columns are added after FORMAT — one per sample.

---

## 📋 Section 3 — The data lines

Each data line is one variant. Let's go through every field in detail.

### Columns 1–5: Position and alleles

| Column | Name | Example | Notes |
|---|---|---|---|
| CHROM | Chromosome | `chr1:100000-101000` | The contig name from the reference |
| POS | Position | `201` | **1-based**. The position of the REF allele in the reference |
| ID | Identifier | `.` | rsID from dbSNP if known, otherwise `.` |
| REF | Reference allele | `A` | The base(s) in the reference at this position |
| ALT | Alternate allele | `T` | The base(s) observed in the sample |

> 💡 For indels, REF and ALT include an **anchor base**. A deletion of `TTT` might appear as `REF=ATTT`, `ALT=A`.

---

### Column 6: QUAL

A Phred-scaled quality score for the variant call:

| QUAL | Probability of error |
|---|---|
| 10 | 1 in 10 chance the call is wrong |
| 20 | 1 in 100 |
| 30 | 1 in 1,000 |
| 50 | 1 in 100,000 |

Higher QUAL = more confident call. In clinical pipelines, a minimum QUAL threshold is usually applied as a filter.

---

### Column 7: FILTER

- `PASS` — the variant passed all filters
- `.` — no filters were applied
- Any other value (e.g. `LowQual`) — the variant failed that filter

In our case, bcftools does not apply filters by default, so you will likely see `.` here.

---

### Column 8: INFO

A semicolon-separated list of key=value pairs with statistics about the variant. Common fields from bcftools:

| Key | Meaning |
|---|---|
| `DP` | Total read depth at this position |
| `VDB` | Variant distance bias (strand bias test) |
| `AF1` | Maximum likelihood allele frequency |
| `MQ` | Mean mapping quality of reads covering this site |
| `FS` | Fisher strand bias score |

---

### Columns 9–10: FORMAT and SAMPLE

These two columns work together. FORMAT defines the order of fields, and SAMPLE provides the values.

Example:
```
FORMAT:   GT:PL:DP:AD
SAMPLE:   0/1:120,0,80:40:20,20
```

| Key | Meaning | Example value |
|---|---|---|
| `GT` | Genotype | `0/1` |
| `DP` | Read depth at this site | `40` |
| `AD` | Allelic depth (ref reads, alt reads) | `20,20` |
| `PL` | Phred-scaled genotype likelihoods | `120,0,80` |

---

## 🧬 Understanding genotype codes

The `GT` field is one of the most important. It describes the genotype of the sample at this position.

Numbers refer to alleles:
- `0` = the REF allele
- `1` = the first ALT allele
- `2` = the second ALT allele (if present)

| GT | Meaning | Clinical term |
|---|---|---|
| `0/0` | Both copies match the reference | Homozygous reference |
| `0/1` | One copy matches reference, one is ALT | Heterozygous |
| `1/1` | Both copies carry the ALT allele | Homozygous alternate |
| `./.` | Genotype could not be determined | No call |

> 👉 What genotype does our synthetic sample show? Is that expected, given how we generated the reads?

---

## 🔎 Exploring our VCF with bcftools

### View only the header

In [ ]:
!bcftools view -h ../fastq_files/variants.vcf

### Extract specific fields with `bcftools query`

This is very useful in practice — you can pull out exactly the columns you need into a clean table:

In [ ]:
!bcftools query -f 'CHROM=%CHROM\tPOS=%POS\tREF=%REF\tALT=%ALT\tQUAL=%QUAL\tGT=%GT\tDP=%DP\n' ../fastq_files/variants.vcf

### Get summary statistics with `bcftools stats`

In [ ]:
!bcftools stats ../fastq_files/variants.vcf | grep -v "^#"

---

## 🏥 VCF in a clinical context

In a clinical diagnostic pipeline, the VCF produced by the variant caller is just the starting point. Before reporting, variants go through several additional steps:

**Filtering** — removing low-quality calls using thresholds on QUAL, DP, AF, strand bias, etc.

**Annotation** — adding biological and clinical meaning to each variant:

| Database | What it adds |
|---|---|
| **Ensembl VEP / ANNOVAR** | Gene name, transcript, predicted consequence (missense, frameshift...) |
| **gnomAD** | Population frequency — is this variant common or rare? |
| **ClinVar** | Clinical significance — has this variant been reported before? |
| **OMIM** | Associated diseases for the gene |

**Interpretation** — a clinical scientist reviews the annotated variants and decides which are clinically significant.

> 💡 The VCF format is designed to carry all this annotation in the INFO column — that is why understanding those key-value pairs is so important.

---

### 🎯 You have now completed the core pipeline:

```
FASTQ → FastQC → BWA → SAM/BAM → bcftools → VCF
```

Every clinical NGS workflow follows this same logic, regardless of the specific tools used.

## About the Author

<p align="center">
  <img src="https://raw.githubusercontent.com/Manuel-DominguezCBG/ngs-teaching-binder/main/assets/Manolo.jpg" width="120" style="border-radius: 6px;">
</p>

**Manuel — Bioinformatician**

Manuel works as a bioinformatician in clinical genomics. He created this resource to support trainees taking their first steps in NGS bioinformatics, with the aim of making the learning process a little less daunting.

Feedback, adaptations, and contributions to this teaching resource are welcome.

**LinkedIn:** [Manuel J. Domínguez](https://www.linkedin.com/in/manuel-j-dom%C3%ADnguez-aa97981b2/)  
**GitHub Repository:** [ngs‑teaching‑binder](https://github.com/Manuel-DominguezCBG/ngs-teaching-binder)

**Huge thanks to the Bioinformatics team and all my WGLS colleagues for helping me make these learning resources better.**